# 0 || Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import numpy as np
import pandas as pd
from pandas.api.types import is_integer_dtype
from pandas.tseries.holiday import USFederalHolidayCalendar

import sklearn
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning

import xgboost
from xgboost import XGBRegressor

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, Dataset


In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
torch 2.9.1


# 1 || Data Processing and Feature Extraction

## 1.1 : Load the DataFrames

In [4]:
# Specify columns to extract.

ECOLS = [
    "UTC time",
    "Demand forecast",
    "Adjusted demand",
    "Adjusted net generation",
    "Adjusted total interchange",
    "FPC",
    "FMPP",
    "SOCO",
    "TEC",
    "JEA",
    "SEC",
    "HST",
    "GVL",
]

WCOLS = [
    "DATE",
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed",
    "HourlyWindDirection",
    "HourlySkyConditions",
]


In [5]:
# Construct raw energy and weather datasets.

DFDICT = {
    subdir.name: { 
        csv.stem: pd.read_csv(csv, dtype="string", low_memory=False)
        for csv in subdir.glob("*.csv") 
    }
    for subdir in (Path.cwd() / "data").iterdir() if subdir.is_dir()
}

DFE = ( 
    DFDICT['electricity']['FPL'][ECOLS]
)
DFW = pd.concat(
    [DFDICT['weather']['MIA_20'+str(y)][WCOLS] for y in range(15, 26)],
    ignore_index=True
)


## 1.2 : Prepocessing and Missing Value Inspection

In [6]:
# Temporal columns are converted to datetime dtypes, adjusted to 
#  local time (UTC-5), and renamed 't'.
# Empty string or single space entries at set to N/A observations.
# Commas are removed from all values in the energy dataset.

DFE = ( DFE
    .assign(t=lambda df: pd.to_datetime(df["UTC time"]) - pd.Timedelta(hours=5))
    .drop(columns=["UTC time"])
    .replace(["", " "], pd.NA)
    .replace(",", "", regex=True)
)

DFW = ( DFW
    .assign(t=lambda df: pd.to_datetime(df["DATE"]))
    .drop(columns=["DATE"])
    .replace(["", " ", "*"], pd.NA)
)

# Drop the records in each dataset that are earlier or later than
#  all records in the other.

DFE = DFE[DFE["t"] <= DFW["t"].iloc[-1].round("d")]
DFW = DFW[DFW["t"] >= DFE["t"].iloc[0].normalize() - pd.Timedelta(days=1)]

# Merge both dataframes, joining electricity records to weather records
#  according to the nearest time measurement.

DF = pd.merge_asof(DFE, DFW, on="t", direction="nearest")


In [7]:
# # Missing values heatmap.
# plt.figure(figsize=(16, 8))
# sns.heatmap(DF[ECOLS[1:]+WCOLS[1:]].isnull().T, cbar=False, cmap='viridis')
# plt.title('Heatmap of Missing Values in the Dataset')
# plt.xlabel('Features')
# plt.ylabel('Missing Values')
# plt.show()

In [8]:
# # List proportions of missing values.
# print("\n\t"+"="*20+" energyFeatures "+"="*20+"\n")
# for col in ECOLS[1:]: 
#     print(f"{col:>35}:" +" "*4 + f"{DF[col].isna().sum()/len(DF):.5f}")
# print("\n\t"+"="*20+" weatherFeatures "+"="*20+"\n")
# for col in WCOLS[1:]: 
#     print(f"{col:>35}:" +" "*4 + f"{DF[col].isna().sum()/len(DF):.5f}")

In [9]:
# Counts of streaks of missing values in each column.
missingStreakDict = { col: {} for col in ECOLS[1:]+WCOLS[1:] }
for col, _dict in missingStreakDict.items(): 
    mask = DF[col].isna()
    for _, run in mask.groupby((mask != mask.shift()).cumsum()):
        if not run.iloc[0]: continue
        _len = int(run.size)
        if _len not in _dict: _dict[_len] = []
        _dict[_len].append(run.index[0])
    width = max(len(f"=== {c} ===") for c in missingStreakDict.keys())
    header = f"=== {col} ===".center(width, "=")
    print(f"\n{header}\n")
    print( pd.Series({_len: len(idxList) for _len, idxList in _dict.items()})
             .sort_index()
             .rename_axis("Missing Streak Length")
             .reset_index(name="Count")
             .to_string(index=False)
    )


======== Demand forecast =========

 Missing Streak Length  Count
                     1     23
                     2      1
                    23      7
                    48      1

======== Adjusted demand =========

 Missing Streak Length  Count
                    24      1

==== Adjusted net generation =====

 Missing Streak Length  Count
                    24      1
                   216      1
                   288      1

=== Adjusted total interchange ===

 Missing Streak Length  Count
                     1      1
                     2      1
                    10      1
                    13      1
                    14      2
                    16      1
                    24      1
                    48      1

============== FPC ===============

 Missing Streak Length  Count
                     1      1
                    23      8
                    24     13
                    25      8
                    48      2
                    71      1

====

In [10]:
# # Count the number of rows with any missing features before any imputation
# #  and after imputing all missing value gaps 1-3 hours long.
# print("Rows with any NA (start):", 
#       DF[ECOLS[1:]+WCOLS[1:-2]].isna().any(axis=1).sum())
# _df = DF.copy()
# for runLen in [1, 2, 3]:
#     for col, run in missingStreakDict.items():
#         if col not in WCOLS[1:-2]: continue
#         if runLen not in run: continue
#         fills = _df[col].ffill().bfill()
#         for runIdx in run[runLen]:
#             runIndices = range(runIdx, runIdx + runLen)
#             _df.loc[runIndices, col] = fills.loc[runIndices]
#     print(f"Rows with any NA (after len={runLen}):",
#           _df[ECOLS[1:]+WCOLS[1:-2]].isna().any(axis=1).sum())

## 1.3 : Imputation and Feature Construction

In [11]:
# # Unique value counts for sharp distributions observed above.
# pd.set_option("display.max_rows", None)
# for col in ["SEC", "HST", "HourlyPrecipitation", "HourlyVisibility", "HourlyWindSpeed"]:
#     print(f"\n=== {col} ===")
#     print(DF[col].value_counts(dropna=False).sort_index())

In [12]:
def plot_delta_distributions(df, cols):
    """
    For each column in `cols`, plot:
      - the value distribution
      - distributions of Δ over 1–4 hours, with σ_Δ / σ annotated.
    """
    
    for col in cols:
        fig, axes = plt.subplots(1, 5, figsize=(20, 3), sharey=True)
        vals = df[col].dropna()
        stdev = vals.std()
        axes[0].hist(vals, bins=30)
        axes[0].set_title(f"{col}\n(σ={stdev:.3g})")

        for i, h in enumerate(range(1, 5), start=1):
            if col == "HourlyWindDirection":
                angles = df[col].astype("float")
                diffs = ((angles - angles.shift(h) + 180) % 360) - 180
                diffs = diffs.dropna()
            else:
                diffs = df[col].diff(periods=h).dropna()
            axes[i].hist(diffs, bins=30)
            axes[i].set_title(
                f"Δ over {h}h\n"
                + "$\\sigma_{\\Delta}/\\sigma \\approx $"
                + f"{diffs.std()/stdev:.4f}"
            )

        fig.suptitle(f"Distributions for {col}")
        plt.tight_layout()

def impute_missing_streaks(df, cols, missingDict, runLengths=(1, 2)):
    """
    For each column in `cols`, impute missing values using interpolation for streaks whose 
     length is in `runLengths`, with special handling for HourlyWindDirection as circular data.
    Prints number of rows with any NA after each run length.
    """
    
    for col in cols:
        
        if col not in missingDict: continue
        runs = missingDict[col]
        dtypeFlag = is_integer_dtype(df[col].dtype)
        
        for _len in runLengths:
            
            if _len not in runs: continue
            startIndices = runs[_len]
            if col == "HourlyWindDirection":
                rads = np.deg2rad(df[col].astype("float"))
                sinRads = np.sin(rads).interpolate(limit_direction="both")
                cosRads = np.cos(rads).interpolate(limit_direction="both")
                interpolations = np.rad2deg(np.arctan2(sinRads, cosRads)) % 360
            else:
                interpolations = df[col].interpolate(limit_direction="both")
            if dtypeFlag: interpolations = interpolations.round()
            
            for idx in startIndices:
                runIndices = range(idx, idx + _len)
                df.loc[runIndices, col] = interpolations.loc[runIndices]
                if col == "HourlyWindDirection":
                    df.loc[runIndices, "HourlyWindDirection_Flag_VRB"] = False


In [13]:
# AUTOREGRESSIVE (LAG) FEATURES

DF["Adjusted demand"] = DF["Adjusted demand"].astype("Int32")
lags = sorted(set(
    list(range(1, 7)) + [12, 18]
    + list(range(24, 27)) + [36, 48]
    + [24*i for i in range(3,7)]
    + [24*7*i for i in range(1,5)]
))
for h in lags: DF[f"Adjusted demand -{h} hr"] = DF["Adjusted demand"].shift(h)
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]


In [14]:
# CALENDAR FEATURES

# Create raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]
DF["Hour"] = DF["t"].dt.hour.astype("Int8")
DF["Month"] = DF["t"].dt.month.astype("Int8")
DF["DayOfWeek"] = DF["t"].dt.dayofweek.astype("Int8")
DF["DayOfYear"] = DF["t"].dt.dayofyear.astype("Int16")

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    rads = np.pi * DF["Hour"] / 12
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    DF[f"sin({argStr})"] = np.sin(rads).astype("float32")
    DF[f"cos({argStr})"] = np.cos(rads).astype("float32")
    rads = np.pi * i * DF["DayOfYear"] / 182.625
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    DF[f"sin({argStr})"] = np.sin(rads).astype("float32")
    DF[f"cos({argStr})"] = np.cos(rads).astype("float32")

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]
DF["Day_Flag_Weekend"] = (DF["DayOfWeek"] >= 5).astype("bool")
DF["Day_Flag_Holiday"] = DF["t"].dt.date.isin(
    USFederalHolidayCalendar().holidays(
        start=DF["t"].min().date(),
        end=DF["t"].max().date()
    ).date
)
DF = pd.concat([ DF,
    pd.get_dummies(DF["Hour"], prefix="Hour_Flag"),
    pd.get_dummies(DF["DayOfWeek"], prefix="DayOfWeek_Flag"),
    pd.get_dummies(DF["Month"], prefix="Month_Flag")
], axis=1)

# Collate all calendar features
calendarFeatures = (
    intDateTimeFeatures
    + hourFourierFeatures
    + dayFourierFeatures
    + hourDummyFeatures
    + dayDummyFeatures
    + monthDummyFeatures
)


In [15]:
# ENERGY FEATURES

# Convert to numeric types.
for i, col in enumerate(ECOLS[1:2]+ECOLS[3:]):
    DF[col] = DF[col].astype("Int16" if i > 1 else "Int32")

# Set extremely anomalous values to null.
DF.loc[DF["SEC"].eq(2465), "SEC"] = pd.NA
DF.loc[DF["HST"].eq(-2278), "HST"] = pd.NA

# Collate energy features. 
energyFeatures = ECOLS[2:]


In [16]:
# WEATHER FEATURES

# Construct indicator flags for observations indicated with
#  suspicious values and variable visibility.
# Remove those alpha characters from those observations.

colsWithFlags = [
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
]

for col in colsWithFlags:
    ch = "V" if (col == "HourlyVisibility") else "s"
    mask = DF[col].str.endswith(ch)
    DF[f"{col}_Flag_{ch}"] = mask
    DF.loc[mask, col] = DF.loc[mask, col].str[:-1]

# Construct indicator flags for sky condition codes.

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'VV']
for code in skyCodes:
    mask = DF["HourlySkyConditions"].str.contains(code, na=False).astype(bool)
    DF[f"HourlySkyConditions_Flag_{code}"] = mask
DF["HourlySkyConditions_Flag_NA"] = DF["HourlySkyConditions"].isna()

# Construct an indicator flag for VRB wind direction observations
#  and set those observations to be N/A.

mask = DF["HourlyWindDirection"].eq("VRB")
DF["HourlyWindDirection_Flag_VRB"] = mask
DF.loc[mask, "HourlyWindDirection"] = pd.NA

# Replace trace precipiataion observations with a reasonable numeric value.

DF["HourlyPrecipitation"] = DF["HourlyPrecipitation"].replace("T", "0.0025")

# Convert columns to appropriate numeric types.

for col in WCOLS[1:-1]:
    if col in colsWithFlags[1:]: DF[col] = DF[col].astype("Float32")
    else: DF[col] = DF[col].astype("Int16")

# Set extremely anomalous values to null.
# Impute missing values.

DF.loc[DF["HourlyWindSpeed"].eq(2237), "HourlyWindSpeed"] = pd.NA
impute_missing_streaks(DF, WCOLS[1:-1], missingStreakDict)

# Construct circular (float) and cardinal (one-hot) encodings from
#  imputed HourlyWindDirection columns.

directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
for i, d in enumerate(directions):
    DF[f"HourlyWindDirection_Flag_{d}"] = (
        (((22.5 + DF["HourlyWindDirection"]) % 360) // 45).astype("Int8").eq(i)
    )
    DF.loc[mask, f"HourlyWindDirection_Flag_{d}"] = False

rads = np.deg2rad(DF["HourlyWindDirection"].astype("Float32"))
DF["sin(HourlyWindDirection)"] = np.sin(rads)
DF["cos(HourlyWindDirection)"] = np.cos(rads)
DF.loc[mask, ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]] = 0.0

# Collate all weather features.

weatherFeatures = (
    WCOLS[1:-1]
    + [col + "_Flag_s" for col in colsWithFlags[:-1]] + ["HourlyVisibility_Flag_V",]
    + [f"HourlySkyConditions_Flag_{code}" for code in [*skyCodes, "NA"]]
    + [f"HourlyWindDirection_Flag_{d}" for d in directions] + ["HourlyWindDirection_Flag_VRB",]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)

# Define other convenient feature variables.

target = "Adjusted demand"
allFeatures = (
    lagFeatures + calendarFeatures 
    + energyFeatures + weatherFeatures
)


In [17]:
# flagCols = [
#     *(col + "_Flag_s" for col in colsWithFlags[:-1]),
#     "HourlyVisibility_Flag_V",
#     *(f"HourlySkyConditions_Flag_{code}" for code in skyCodes),
#     "HourlyWindDirection_Flag_VRB",
#     *(f"HourlyWindDirection_Flag_{d}" for d in directions),
# ]

# flagCounts = DF[flagCols].agg([
#     "sum", 
#     lambda x: (~x).sum(), 
#     lambda x: x.isna().sum()
# ]).T

# flagCounts.columns = ["True", "False", "NaN"]
# print(flagCounts)


In [18]:
# plot_delta_distributions(DF, WCOLS[1:-1])

In [19]:
DF = DF.drop(
    [
        *(col + "_Flag_s" for col in colsWithFlags[:-1]), 
        "HourlyVisibility_Flag_V", 
        "HourlySkyConditions_Flag_VV",
        "HourlyWindDirection",
        "HourlySkyConditions",
    ],
    axis=1
)
weatherFeatures = [c for c in weatherFeatures if c in DF.columns]

In [20]:
# # Counts of streaks of missing values in each column.
# checkFeatures = ECOLS[1:]+WCOLS[1:-2]+["HourlyWindDirection_Flag_VRB"]
# checkStreakDict = { col: {} for col in checkFeatures }
# for col, _dict in checkStreakDict.items(): 
#     mask = DF[col].isna()
#     for _, run in mask.groupby((mask != mask.shift()).cumsum()):
#         if not run.iloc[0]: continue
#         _len = int(run.size)
#         if _len not in _dict: _dict[_len] = []
#         _dict[_len].append(run.index[0])
#     width = max(len(f"=== {c} ===") for c in checkStreakDict.keys())
#     header = f"=== {col} ===".center(width, "=")
#     print(f"\n{header}\n")
#     print( pd.Series({_len: len(idxList) for _len, idxList in _dict.items()})
#              .sort_index()
#              .rename_axis("Missing Streak Length")
#              .reset_index(name="Count")
#              .to_string(index=False)
#     )

In [21]:
# # Missing values heatmap.
# plt.figure(figsize=(16, 8))
# sns.heatmap(DF.isnull().T, cbar=False, cmap='viridis')
# plt.title('Heatmap of Missing Values in the Dataset')
# plt.xlabel('Features')
# plt.ylabel('Missing Values')
# plt.show()

In [22]:
print(DF.shape); DF = DF.dropna(); print(DF.shape)
print(min(DF['t']), max(DF['t']))

(89017, 118)
(85211, 118)
2015-07-29 00:00:00 2025-08-26 00:00:00


In [23]:
# pd.set_option("display.max_rows", None)
# timeDeltas = DF["t"].diff()
# timeGroups = (timeDeltas != pd.Timedelta("1h")).cumsum()
# seqs = DF.groupby(timeGroups)["t"].agg(start="min", end="max", n="size")
# seqs = seqs[seqs["n"] > 1][["start", "end"]]
# print(seqs)

In [24]:
for col in DF.columns: print(col, DF[col].dtype)

Demand forecast Int32
Adjusted demand Int32
Adjusted net generation Int32
Adjusted total interchange Int16
FPC Int16
FMPP Int16
SOCO Int16
TEC Int16
JEA Int16
SEC Int16
HST Int16
GVL Int16
t datetime64[ns]
HourlyDryBulbTemperature Int16
HourlyPrecipitation Float32
HourlyRelativeHumidity Int16
HourlySeaLevelPressure Float32
HourlyVisibility Float32
HourlyWindSpeed Int16
Adjusted demand -1 hr Int32
Adjusted demand -2 hr Int32
Adjusted demand -3 hr Int32
Adjusted demand -4 hr Int32
Adjusted demand -5 hr Int32
Adjusted demand -6 hr Int32
Adjusted demand -12 hr Int32
Adjusted demand -18 hr Int32
Adjusted demand -24 hr Int32
Adjusted demand -25 hr Int32
Adjusted demand -26 hr Int32
Adjusted demand -36 hr Int32
Adjusted demand -48 hr Int32
Adjusted demand -72 hr Int32
Adjusted demand -96 hr Int32
Adjusted demand -120 hr Int32
Adjusted demand -144 hr Int32
Adjusted demand -168 hr Int32
Adjusted demand -336 hr Int32
Adjusted demand -504 hr Int32
Adjusted demand -672 hr Int32
Hour Int8
Month Int

In [25]:
print(len(
    lagFeatures
    + calendarFeatures
    + energyFeatures
    + weatherFeatures
))

print([
    c for c in DF.columns if c not in
    lagFeatures + calendarFeatures + energyFeatures + weatherFeatures
])

116
['Demand forecast', 't']


### 1.3 Data Splits

In [26]:
trainIdx = int(len(DF) * 0.7)
valIdx = int(len(DF) * 0.85)
DFtrain = DF[:trainIdx]
DFval = DF[trainIdx:valIdx]
DFtest =  DF[valIdx:]

In [27]:
path = Path("data/clean")
path.mkdir(parents=True, exist_ok=True)
DF.to_pickle(path/"DF.pkl")

In [28]:
pd.set_option("display.max_rows", None)

path = Path("data/clean/train")
path.mkdir(parents=True, exist_ok=True)
DFtrain.to_pickle(path/"DFtrain.pkl")

timeGroups = (DFtrain["t"].diff() != pd.Timedelta("1h")).cumsum()
for g, block in DFtrain.groupby(timeGroups):
    block.to_pickle(path/f"DFtrain_block{str(g).zfill(3)}.pkl")
    print(g, block["t"].iloc[0], block["t"].iloc[-1], len(block))

1 2015-07-29 00:00:00 2015-08-02 23:00:00 120
2 2015-08-03 14:00:00 2015-08-11 23:00:00 202
3 2015-08-12 14:00:00 2015-08-28 21:00:00 392
4 2015-08-29 02:00:00 2015-10-31 23:00:00 1534
5 2015-11-02 01:00:00 2015-11-09 23:00:00 191
6 2015-11-10 17:00:00 2015-11-15 21:00:00 125
7 2015-11-16 02:00:00 2015-11-27 00:00:00 263
8 2015-11-28 12:00:00 2015-12-14 23:00:00 396
9 2015-12-15 03:00:00 2015-12-30 00:00:00 358
10 2015-12-31 02:00:00 2016-01-01 00:00:00 23
11 2016-01-02 01:00:00 2016-01-03 23:00:00 47
12 2016-01-04 03:00:00 2016-01-12 23:00:00 213
13 2016-01-13 17:00:00 2016-01-28 23:00:00 367
14 2016-01-29 03:00:00 2016-02-01 21:00:00 91
15 2016-02-02 02:00:00 2016-02-25 00:00:00 551
16 2016-02-26 01:00:00 2016-03-11 00:00:00 336
17 2016-03-14 00:00:00 2016-04-12 23:00:00 720
18 2016-04-14 00:00:00 2016-06-23 13:00:00 1694
19 2016-06-24 00:00:00 2016-06-24 21:00:00 22
20 2016-06-25 00:00:00 2016-08-29 21:00:00 1582
21 2016-08-30 02:00:00 2016-09-08 06:00:00 221
22 2016-09-08 11:00:00 

In [29]:
path = Path("data/clean/val")
path.mkdir(parents=True, exist_ok=True)
DFval.to_pickle(path/"DFval.pkl")

timeGroups = (DFval["t"].diff() != pd.Timedelta("1h")).cumsum()
for g, block in DFval.groupby(timeGroups):
    block.to_pickle(path/f"DFval_block{str(g).zfill(3)}.pkl")
    print(g, block["t"].iloc[0], block["t"].iloc[-1], len(block))

1 2022-07-29 00:00:00 2022-08-03 08:00:00 129
2 2022-08-03 12:00:00 2022-08-06 21:00:00 82
3 2022-08-07 02:00:00 2022-09-09 13:00:00 804
4 2022-09-09 17:00:00 2022-09-17 21:00:00 197
5 2022-09-18 02:00:00 2022-09-22 17:00:00 112
6 2022-09-24 11:00:00 2022-10-01 23:00:00 181
7 2022-10-02 06:00:00 2022-10-02 06:00:00 1
8 2022-10-03 02:00:00 2022-10-03 18:00:00 17
9 2022-10-05 02:00:00 2022-10-05 21:00:00 20
10 2022-10-06 02:00:00 2022-10-06 21:00:00 20
11 2022-10-07 02:00:00 2022-10-20 23:00:00 334
12 2022-10-21 03:00:00 2022-11-05 23:00:00 381
13 2022-11-07 01:00:00 2022-11-10 23:00:00 95
14 2022-11-11 03:00:00 2022-11-23 05:00:00 291
15 2022-11-23 11:00:00 2022-12-17 06:00:00 572
16 2022-12-17 11:00:00 2023-03-12 00:00:00 2030
17 2023-03-13 00:00:00 2023-03-13 22:00:00 23
18 2023-03-14 00:00:00 2023-05-19 05:00:00 1590
19 2023-05-19 09:00:00 2023-06-12 23:00:00 591
20 2023-06-13 03:00:00 2023-09-20 21:00:00 2395
21 2023-09-21 02:00:00 2023-10-13 21:00:00 548
22 2023-10-14 02:00:00 2023

In [30]:
path = Path("data/clean/test")
path.mkdir(parents=True, exist_ok=True)
DFtest.to_pickle(path/"DFtest.pkl")

timeGroups = (DFtest["t"].diff() != pd.Timedelta("1h")).cumsum()
for g, block in DFtest.groupby(timeGroups):
    block.to_pickle(path/f"DFtest_block{str(g).zfill(3)}.pkl")
    print(g, block["t"].iloc[0], block["t"].iloc[-1], len(block))

1 2024-02-03 19:00:00 2024-03-02 23:00:00 677
2 2024-03-03 03:00:00 2024-03-06 21:00:00 91
3 2024-03-07 02:00:00 2024-03-10 00:00:00 71
4 2024-03-11 00:00:00 2024-03-11 22:00:00 23
5 2024-03-12 00:00:00 2024-03-26 23:00:00 360
6 2024-03-27 03:00:00 2024-05-28 23:00:00 1509
7 2024-06-01 00:00:00 2024-07-01 09:00:00 730
8 2024-07-01 14:00:00 2024-08-02 23:00:00 778
9 2024-08-03 03:00:00 2024-08-14 09:00:00 271
10 2024-08-14 13:00:00 2024-08-27 21:00:00 321
11 2024-08-28 02:00:00 2024-08-31 09:00:00 80
12 2024-08-31 14:00:00 2024-10-07 20:00:00 895
13 2024-10-08 02:00:00 2024-11-02 23:00:00 622
14 2024-11-04 01:00:00 2024-11-30 23:00:00 647
15 2024-12-06 09:00:00 2024-12-06 09:00:00 1
16 2024-12-11 20:00:00 2024-12-11 20:00:00 1
17 2024-12-14 14:00:00 2024-12-14 14:00:00 1
18 2024-12-14 23:00:00 2024-12-15 05:00:00 7
19 2024-12-15 12:00:00 2024-12-15 12:00:00 1
20 2024-12-15 17:00:00 2024-12-15 17:00:00 1
21 2024-12-16 00:00:00 2024-12-16 08:00:00 9
22 2024-12-16 14:00:00 2024-12-17 03:00